In [2]:
from tqdm.auto import tqdm

import jax
from jax import random, numpy as jnp
from flax import nnx
import optax

/Users/jorgvt/Developer/MultiTask/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datasets import load_dataset

In [4]:
dst = load_dataset("Jorgvt/TID2008")

In [6]:
def obtain_dmos(sample):
    sample["dmos"] = 10 - sample["mos"]
    return sample

In [7]:
dst = dst.map(obtain_dmos)

Map: 100%|██████████| 1700/1700 [00:04<00:00, 352.80 examples/s]


In [8]:
dst_train = dst["train"].with_format("jax")

In [13]:
def preprocess(row, resize_to=None):
    img, dist, mos, dmos = row.values()
    img = img/255.
    dist = dist/255.
    if resize_to is not None:
        img = jax.image.resize(img, shape=resize_to, method="linear")
        dist = jax.image.resize(img, shape=resize_to, method="linear")
    return {"reference": img, "distorted": dist, "mos": mos, "dmos": dmos}

In [143]:
# dst_train_rdy = dst_train.map(lambda x: preprocess(x, resize_to=(256,256,3)), num_proc=8)

In [144]:
def pearson_correlation(vec1, vec2):
    vec1 = vec1.squeeze()
    vec2 = vec2.squeeze()
    vec1_mean = vec1.mean()
    vec2_mean = vec2.mean()
    num = vec1 - vec1_mean
    num *= vec2 - vec2_mean
    num = num.sum()
    denom = jnp.sqrt(jnp.sum((vec1 - vec1_mean) ** 2))
    denom *= jnp.sqrt(jnp.sum((vec2 - vec2_mean) ** 2))
    return num / denom

In [145]:
def mse(a, b):
    return ((a-b)**2).mean()**(1/2)

In [146]:
class Model(nnx.Module):
    def __init__(self, *, rngs):
        self.layers = [
            nnx.Conv(in_features=3, out_features=32, kernel_size=5, padding="SAME", rngs=rngs),
            nnx.Conv(in_features=32, out_features=64, kernel_size=3, padding="SAME", rngs=rngs),
            nnx.Conv(in_features=64, out_features=128, kernel_size=3, padding="SAME", rngs=rngs),
        ]
        self.sigma_c = nnx.Param((1.))
        self.sigma_r = nnx.Param((1.))

    def __call__(self, inputs, **kwargs):
        outputs = inputs
        for layer in self.layers:
            outputs = layer(outputs)
            outputs = nnx.relu(outputs)
            outputs = nnx.max_pool(outputs, window_shape=(2,2), strides=(2,2))
        return outputs, self.sigma_c, self.sigma_r

In [147]:
@nnx.jit
def train_step(model, optimizer, img, dist, mos, **kwargs):
    def loss_fn(model, **kwargs):
        img_pred, sigma_c, sigma_r = model(img)
        dist_pred, sigma_c, sigma_r = model(dist)
        sigma_c, sigma_r = jnp.exp(sigma_c), jnp.exp(sigma_r)
        distance = ((img_pred-dist_pred)**2).mean(axis=(1,2,3))**(1/2)
        ## Calculate losses
        corr = pearson_correlation(distance, mos)
        reg = mse(distance, mos)
        return (1/sigma_c)*corr + (1/sigma_r)*reg + jnp.log(sigma_c) + jnp.log(sigma_r), (corr, reg)
    (loss, (corr, reg)), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(grads)
    return loss, corr, reg

In [148]:
model = Model(rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

In [14]:
def preprocess(row, resize_to=None):
    img, dist, mos, dmos = row.values()
    img = img/255.
    dist = dist/255.
    if resize_to is not None:
        img = jax.image.resize(img, shape=resize_to, method="linear")
        dist = jax.image.resize(dist, shape=resize_to, method="linear")
    return img, dist, dmos

In [151]:
epochs = 5
losses = []
losses_c, losses_r = [], []
for epoch in tqdm(range(epochs), desc="Epoch"):
    losses_b, losses_b_c, losses_b_r = [], [], []
    for batch in tqdm(dst_train.iter(batch_size=32, drop_last_batch=True), leave=False, desc="Batches"):
        img, dist, mos = preprocess(batch, resize_to=(32,256,256,3))
        loss, loss_c, loss_r = train_step(model, optimizer, img, dist, mos)
        losses_b.append(loss)
        losses_b_c.append(loss_c)
        losses_b_r.append(loss_r)
        # break
    losses.append(jnp.mean(jnp.array(losses_b)))
    losses_c.append(jnp.mean(jnp.array(losses_b_c)))
    losses_r.append(jnp.mean(jnp.array(losses_b_r)))
    print(f"Epoch {epoch} --> Loss: {losses[-1]} | Corr: {losses_c[-1]} ({model.sigma_c.value:.2f})| Reg: {losses_r[-1]} ({model.sigma_r.value:.2f})")
    # break

Epoch:  20%|██        | 1/5 [03:39<14:37, 219.48s/it]

Epoch 0 --> Loss: 3.377305030822754 | Corr: -0.7685240507125854 (0.95)| Reg: 4.6524481773376465 (1.05)


Epoch:  40%|████      | 2/5 [07:11<10:44, 214.82s/it]

Epoch 1 --> Loss: 3.235888719558716 | Corr: -0.8715909719467163 (0.89)| Reg: 4.655050277709961 (1.10)


Epoch:  60%|██████    | 3/5 [10:22<06:48, 204.38s/it]

Epoch 2 --> Loss: 3.129556179046631 | Corr: -0.8884382843971252 (0.84)| Reg: 4.656317234039307 (1.14)


Epoch:  80%|████████  | 4/5 [13:55<03:27, 207.72s/it]

Epoch 3 --> Loss: 3.031493902206421 | Corr: -0.8919953107833862 (0.78)| Reg: 4.65701150894165 (1.19)


Epoch:  80%|████████  | 4/5 [15:35<03:53, 233.91s/it]


KeyboardInterrupt: 